# Junction Tree Variational Autoencoder for Molecular Graph Generation
## A cluster-oriented reproduction of Jin, Barzilay & Jaakkola (2018)

This notebook explains and reproduces **Junction Tree Variational Autoencoder for Molecular Graph Generation** (JT-VAE), using the authors' implementation for the chemically constrained decoder and fragment assembly.

- [Paper](https://arxiv.org/abs/1802.04364)
- [Authors' ICML 2018 repository](https://github.com/wengong-jin/icml18-jtnn)

All molecular inputs, vocabularies, preprocessing shards, generated SMILES, and run manifests are kept under `data/JunctionTreeVAE/`. Training checkpoints and logs are kept under `models/JunctionTreeVAE/`. Existing checkpoints are reused unless `FORCE_RETRAIN=True`.

## 1. Why generate a junction tree first?

Generating a molecular graph atom by atom creates many partial graphs that violate valence or ring constraints. JT-VAE instead decomposes a molecule into chemically valid clusters—rings and bonds—and connects overlapping clusters as a tree. It then generates in two stages:

1. decode a **junction tree** whose nodes are substructures from a learned vocabulary;
2. assemble neighboring substructures while scoring only chemically valid attachment candidates.

This coarse-to-fine construction builds strong chemical bias into the decoder. It is the main reason this reproduction uses the authors' assembly code rather than a cosmetically similar generic tree VAE.

## 2. Model and objective

A molecule $G$ and its junction tree $T_G$ are encoded separately:

$$q(z_T,z_G\mid T_G,G)=q(z_T\mid T_G)q(z_G\mid G).$$

The tree latent variable controls the scaffold of clusters; the graph latent variable carries fine attachment information. The decoder factorizes as

$$p(T_G,G\mid z_T,z_G)=p(T_G\mid z_T)\,p(G\mid T_G,z_G).$$

Training combines tree topology/label prediction, graph assembly prediction, and KL regularization. The reparameterization trick permits end-to-end gradients. During assembly, candidate mappings between shared atoms are enumerated and invalid candidates are rejected by chemical constraints.

## 3. Cluster environment

Run the notebook inside the GPU allocation—not on a login node. The reference implementation targets Python 2.7, but its tensor code already contains the essential modern PyTorch API fixes. The setup cell makes a private vendor checkout and applies the standard library's `2to3` syntax conversion once, leaving the model logic unchanged. RDKit and CUDA-enabled PyTorch must be installed in the active environment.

The notebook records the exact Git commit and configuration in `data/JunctionTreeVAE/run_manifest.json`. This matters because the upstream repository is external source code, not a versioned dependency in this project.

In [ ]:
import csv
import json
import os
import random
import shutil
import subprocess
import sys
import time
from pathlib import Path

import torch

SEED = 17
random.seed(SEED); torch.manual_seed(SEED)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'JunctionTreeVariationalAutoencoder' else Path.cwd()
NOTEBOOK_DIR = PROJECT_ROOT / 'JunctionTreeVariationalAutoencoder'
DATA_DIR = PROJECT_ROOT / 'data' / 'JunctionTreeVAE'
MODEL_DIR = PROJECT_ROOT / 'models' / 'JunctionTreeVAE'
REPO_DIR = NOTEBOOK_DIR / 'vendor' / 'icml18-jtnn'
DATA_DIR.mkdir(parents=True, exist_ok=True); MODEL_DIR.mkdir(parents=True, exist_ok=True)
if not torch.cuda.is_available():
    print('WARNING: CUDA is unavailable. Setup/decomposition can run, but full training should use a GPU allocation.')
else:
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {p.name} | VRAM: {p.total_memory/2**30:.1f} GiB | PyTorch {torch.__version__}')

In [ ]:
def run(command, cwd=None, env=None, log_path=None):
    command = [str(x) for x in command]
    print('$', ' '.join(command))
    process = subprocess.Popen(command, cwd=cwd, env=env, text=True,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
    log_file = open(log_path, 'a', encoding='utf-8') if log_path else None
    try:
        for line in process.stdout:
            print(line, end='')
            if log_file: log_file.write(line); log_file.flush()
    finally:
        if log_file: log_file.close()
    code = process.wait()
    if code != 0: raise subprocess.CalledProcessError(code, command)

def output(command, cwd=None):
    return subprocess.check_output([str(x) for x in command], cwd=cwd, text=True).strip()

## 4. Acquire and install the reference implementation

The clone lives under the notebook folder and is ignored by Git. `PINNED_COMMIT` may be set to a specific upstream commit hash for strict reproducibility. When left as `None`, the current default-branch revision is recorded in the manifest. Python-3 conversion is applied only inside this private checkout and marked by `.python3_compatible`.

In [ ]:
REPO_URL = 'https://github.com/wengong-jin/icml18-jtnn.git'
PINNED_COMMIT = 'd4c425a8a28fce7d0e8f5da741d27d3fce4a7ec0'
if not REPO_DIR.exists():
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    run(['git', 'clone', REPO_URL, REPO_DIR])
if PINNED_COMMIT:
    run(['git', 'checkout', PINNED_COMMIT], cwd=REPO_DIR)
REPO_COMMIT = output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR)
print('Reference implementation commit:', REPO_COMMIT)
PY3_MARKER = REPO_DIR / '.python3_compatible'
if not PY3_MARKER.exists():
    run([sys.executable, '-m', 'lib2to3', '-w', '-n', REPO_DIR/'fast_jtnn', REPO_DIR/'fast_molvae'])
    preprocess_file = REPO_DIR/'fast_molvae'/'preprocess.py'
    text = preprocess_file.read_text(encoding='utf-8').replace('(len(all_data) + num_splits - 1) / num_splits', '(len(all_data) + num_splits - 1) // num_splits')
    preprocess_file.write_text(text, encoding='utf-8')
    datautils_file = REPO_DIR/'fast_jtnn'/'datautils.py'
    text = datautils_file.read_text(encoding='utf-8')
    text = text.replace('[fn for fn in os.listdir(data_folder)]', '[fn for fn in os.listdir(data_folder) if fn.endswith(".pkl")]')
    text = text.replace('with open(fn) as f:', "with open(fn, 'rb') as f:")
    datautils_file.write_text(text, encoding='utf-8')
    PY3_MARKER.write_text(f'2to3 conversion of {REPO_COMMIT}\n', encoding='utf-8')
if str(REPO_DIR) not in sys.path: sys.path.insert(0, str(REPO_DIR))
SUBPROCESS_ENV = os.environ.copy()
SUBPROCESS_ENV['PYTHONPATH'] = str(REPO_DIR) + os.pathsep + SUBPROCESS_ENV.get('PYTHONPATH', '')
import fast_jtnn
from rdkit import Chem, rdBase
print('RDKit:', rdBase.rdkitVersion)

## 5. Persist the ZINC training set and vocabulary

The authors distribute the filtered ZINC training SMILES and substructure vocabulary with the repository. This cell copies them into the project's canonical `data/JunctionTreeVAE/` folder. If your checkout does not contain them, place equivalent files there as `train.txt` and `vocab.txt`. Never rebuild a vocabulary using validation/test molecules.

In [ ]:
TRAIN_PATH = DATA_DIR / 'train.txt'
VOCAB_PATH = DATA_DIR / 'vocab.txt'
for filename, destination in [('train.txt', TRAIN_PATH), ('vocab.txt', VOCAB_PATH)]:
    source = REPO_DIR / 'data' / 'zinc' / filename
    if not destination.exists():
        if not source.exists():
            raise FileNotFoundError(f'Provide {destination}; it was not present in the reference checkout.')
        shutil.copy2(source, destination)
with TRAIN_PATH.open(encoding='utf-8') as f: smiles = [line.strip() for line in f if line.strip()]
with VOCAB_PATH.open(encoding='utf-8') as f: vocabulary = [line.strip() for line in f if line.strip()]
invalid = sum(Chem.MolFromSmiles(s) is None for s in smiles[:5000])
print(f'Training molecules: {len(smiles):,}; vocabulary: {len(vocabulary):,}; invalid in first 5k: {invalid}')

## 6. Inspect a molecular junction tree

`MolTree` applies the paper's ring/bond clique decomposition and maximum-spanning-tree construction. Each displayed node is a valid molecular substructure; edges indicate atom overlap and legal assembly relationships.

In [ ]:
from fast_jtnn import MolTree
example_smiles = 'CC1=CC=C(C=C1)C(=O)NC2=CC=CC=C2'
tree = MolTree(example_smiles)
print('Molecule:', example_smiles)
for i, node in enumerate(tree.nodes):
    neighbor_ids = [tree.nodes.index(n) for n in node.neighbors]
    print(f'{i:2d}: {node.smiles:20s} neighbors={neighbor_ids}')

## 7. Parallel preprocessing

Tree decomposition and assembly-candidate enumeration are CPU/RDKit work. They are performed once in parallel and cached as tensor shards in `data/JunctionTreeVAE/processed/`. GPU training then streams these reusable shards.

In [ ]:
PROCESSED_DIR = DATA_DIR / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
PREPROCESS_MARKER = PROCESSED_DIR / '.complete'
PREPROCESS_JOBS = min(16, os.cpu_count() or 1)
N_SHARDS = 100
if PREPROCESS_MARKER.exists() and f'commit={REPO_COMMIT}' not in PREPROCESS_MARKER.read_text(encoding='utf-8'):
    raise ValueError('Preprocessed shards came from a different source commit; remove processed/ and rebuild.')
if not PREPROCESS_MARKER.exists():
    run([sys.executable, REPO_DIR/'fast_molvae'/'preprocess.py', '--train', TRAIN_PATH,
         '--split', N_SHARDS, '--jobs', PREPROCESS_JOBS], cwd=PROCESSED_DIR,
        env=SUBPROCESS_ENV, log_path=MODEL_DIR/'preprocess.log')
    shards = list(PROCESSED_DIR.glob('tensors-*.pkl'))
    if len(shards) != N_SHARDS: raise RuntimeError(f'Expected {N_SHARDS} shards, found {len(shards)}')
    PREPROCESS_MARKER.write_text(f'commit={REPO_COMMIT}\nshards={N_SHARDS}\n', encoding='utf-8')
print(f'Preprocessed shards: {len(list(PROCESSED_DIR.glob("tensors-*.pkl")))}')

## 8. Train the full JT-VAE

These are the reference-scale architectural settings: hidden size 450, total latent size 56, tree depth 20, graph depth 3, batch size 40, gradient clipping, learning-rate annealing, and KL annealing. Checkpoints are written directly into `models/JunctionTreeVAE/`.

The original script controls its own CUDA execution and checkpoint format, so this notebook deliberately does not wrap it in an unrelated AMP training loop. Set `CUDA_VISIBLE_DEVICES` before starting Jupyter when selecting a cluster GPU.

In [ ]:
FORCE_RETRAIN = False
CONTINUE_TRAINING = False  # resume from the latest checkpoint instead of only loading it
TRAIN_CONFIG = dict(hidden_size=450, latent_size=56, depthT=20, depthG=3, beta=0, lr=1e-3,
                    clip_norm=50, epoch=20, batch_size=40, anneal_rate=0.9,
                    anneal_iter=40000, kl_anneal_iter=2000, warmup=40000)
MANIFEST_PATH = DATA_DIR / 'run_manifest.json'
manifest = {'repository': REPO_URL, 'commit': REPO_COMMIT, 'train_config': TRAIN_CONFIG,
            'pytorch': torch.__version__, 'cuda': torch.version.cuda}
checkpoints = sorted(MODEL_DIR.glob('model.iter-*'), key=lambda p: int(p.name.rsplit('-',1)[-1]))
if checkpoints and not FORCE_RETRAIN and not CONTINUE_TRAINING:
    if not MANIFEST_PATH.exists() or json.loads(MANIFEST_PATH.read_text(encoding='utf-8')) != manifest:
        raise ValueError('Checkpoint manifest differs from this source/configuration; set FORCE_RETRAIN=True.')
    print(f'Reusing latest checkpoint: {checkpoints[-1]}')
else:
    if checkpoints and CONTINUE_TRAINING and (not MANIFEST_PATH.exists() or json.loads(MANIFEST_PATH.read_text(encoding='utf-8')) != manifest):
        raise ValueError('Cannot resume: checkpoint manifest differs from this source/configuration.')
    if FORCE_RETRAIN:
        for old_checkpoint in checkpoints: old_checkpoint.unlink()
        checkpoints = []
    MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    if not torch.cuda.is_available(): raise RuntimeError('Full JT-VAE training requires a CUDA allocation.')
    c = TRAIN_CONFIG
    command = [sys.executable, REPO_DIR/'fast_molvae'/'vae_train.py', '--train', PROCESSED_DIR,
               '--vocab', VOCAB_PATH, '--save_dir', MODEL_DIR, '--hidden_size', c['hidden_size'],
               '--latent_size', c['latent_size'], '--depthT', c['depthT'], '--depthG', c['depthG'],
               '--beta', c['beta'], '--lr', c['lr'], '--clip_norm', c['clip_norm'],
               '--epoch', c['epoch'], '--batch_size', c['batch_size'],
               '--anneal_rate', c['anneal_rate'], '--anneal_iter', c['anneal_iter'],
               '--kl_anneal_iter', c['kl_anneal_iter'], '--warmup', c['warmup']]
    if checkpoints and CONTINUE_TRAINING:
        command.extend(['--load_epoch', int(checkpoints[-1].name.rsplit('-', 1)[-1])])
    run(command, cwd=REPO_DIR/'fast_molvae', env=SUBPROCESS_ENV, log_path=MODEL_DIR/'training.log')
    checkpoints = sorted(MODEL_DIR.glob('model.iter-*'), key=lambda p: int(p.name.rsplit('-',1)[-1]))
if not checkpoints: raise FileNotFoundError('Training produced no model.iter-* checkpoint.')
CHECKPOINT_PATH = checkpoints[-1]
print('Checkpoint:', CHECKPOINT_PATH)

## 9. Prior sampling and chemical evaluation

The reference sampler draws latent vectors, decodes junction trees, enumerates attachments, and returns assembled SMILES. Generated strings are persisted in `data/JunctionTreeVAE/generated_smiles.txt`. RDKit then measures validity, uniqueness among valid molecules, and novelty against the training set.

In [ ]:
N_SAMPLES = 1000
GENERATED_PATH = DATA_DIR / 'generated_smiles.txt'
if not GENERATED_PATH.exists():
    c = TRAIN_CONFIG
    command = [sys.executable, REPO_DIR/'fast_molvae'/'sample.py', '--nsample', N_SAMPLES,
               '--vocab', VOCAB_PATH, '--model', CHECKPOINT_PATH, '--hidden_size', c['hidden_size'],
               '--latent_size', c['latent_size'], '--depthT', c['depthT'], '--depthG', c['depthG']]
    result = subprocess.run([str(x) for x in command], cwd=REPO_DIR/'fast_molvae',
                            text=True, capture_output=True, check=True, env=SUBPROCESS_ENV)
    GENERATED_PATH.write_text(result.stdout, encoding='utf-8')
generated = [line.strip() for line in GENERATED_PATH.read_text(encoding='utf-8').splitlines() if line.strip()]
valid = []
for s in generated:
    mol = Chem.MolFromSmiles(s)
    if mol is not None: valid.append(Chem.MolToSmiles(mol))
training_canonical = {Chem.MolToSmiles(m) for s in smiles for m in [Chem.MolFromSmiles(s)] if m is not None}
unique = set(valid)
metrics = {'samples': len(generated), 'validity': len(valid)/max(1,len(generated)),
           'uniqueness': len(unique)/max(1,len(valid)),
           'novelty': len(unique-training_canonical)/max(1,len(unique))}
print(json.dumps(metrics, indent=2))
print('Examples:', list(unique)[:10])

## 10. What this reproduction establishes

This workflow reproduces the paper's defining mechanism: vocabulary-based clique decomposition, dual tree/graph latent representations, autoregressive tree decoding, and chemically constrained graph assembly. It also preserves preprocessing and weights so subsequent sessions do not repeat expensive work.

Reported metrics should still be interpreted carefully. They depend on the exact source commit, ZINC filtering, vocabulary, checkpoint iteration, sampling temperature/implementation, and canonicalization. A scientifically comparable result requires matching the paper's held-out set and evaluation protocol, running multiple seeds, and reporting confidence intervals.

The upstream implementation predates current PyTorch releases. If an incompatibility appears, record any compatibility patch alongside the manifest rather than silently changing model logic. The paper's principal limitation also remains: generation is restricted to combinations of substructures represented in the vocabulary, and assembly search becomes expensive for highly connected motifs.

## Reference

Jin W, Barzilay R, Jaakkola T. *Junction Tree Variational Autoencoder for Molecular Graph Generation.* Proceedings of ICML, PMLR 80:2323–2332, 2018.